In [1]:
# 1. Import libraries
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# 2. Load dataset
df = pd.read_csv("fightCards.csv")
print(df.head())
print(df.shape)

# 3. Clean text
df["text"] = df["text"].fillna("").astype(str)

# 4. NLP: convert text to TF-IDF
tfidf = TfidfVectorizer(stop_words="english")
X = tfidf.fit_transform(df["text"])

# 5. Find best K
scores = []
for k in range(2, 10):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X)
    scores.append(silhouette_score(X, labels))

plt.plot(range(2, 10), scores, marker="o")
plt.xlabel("K")
plt.ylabel("Silhouette Score")
plt.show()

# 6. K-Means clustering
k = 3
model = KMeans(n_clusters=k, random_state=42, n_init=10)
df["cluster"] = model.fit_predict(X)

# 7. Show clusters
print(df[["text", "cluster"]].head(20))
print(df["cluster"].value_counts())

# 8. Important words in each cluster
words = tfidf.get_feature_names_out()

for i in range(k):
    top = model.cluster_centers_[i].argsort()[-10:][::-1]
    print(f"Cluster {i}:", [words[j] for j in top])

# 9. Visualise clusters
pca = PCA(n_components=2)
X2 = pca.fit_transform(X.toarray())

plt.scatter(X2[:,0], X2[:,1], c=df["cluster"])
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.title("Fight Cards K-Means Clusters")
plt.show()

# 10. Save results
df.to_csv("fightCards_clustered.csv", index=False)

                              card_name               f1 f1_sig_strike_per  \
0      UFC Fight Night: Gane vs. Volkov     Charles Rosa               28%   
1      UFC Fight Night: Gane vs. Volkov   Damir Hadzovic               47%   
2   UFC Fight Night: Font vs. Garbrandt  Damir Ismagulov               47%   
3      UFC Fight Night: Gane vs. Volkov      Julia Avila               52%   
4  UFC Fight Night: Hall vs. Strickland     Jinh Yu Frey               47%   

   f1_sig_strike_total  f1_td_attempt  f1_td_succeed                  f2  \
0                  182              2              2       Justin Jaynes   
1                  219              2              2      Yancy Medeiros   
2                   63              1              0        Rafael Alves   
3                   91              4              1  Julija Stoliarenko   
4                  185              1              0        Ashley Yoder   

  f2_sig_strike_per  f2_sig_strike_total  f2_td_attempt  f2_td_succeed  \


KeyError: 'text'